# Gerador SD 2.1 — servidor para o APK

Este notebook executa o código **Stable Diffusion 2.1-base com safety checker obrigatório** e cria um endereço temporário para colar no aplicativo Android.

Antes de começar, escolha **Ambiente de execução → Alterar tipo de ambiente de execução → GPU**, se a opção estiver disponível. Depois use **Executar tudo**.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = Path('/content/novaforge-apk')
if not REPO.exists():
    subprocess.check_call([
        'git', 'clone', '--depth', '1', '--branch', 'sd21-exact-app',
        'https://github.com/danielchile981-art/novaforge-apk.git', str(REPO)
    ])
else:
    subprocess.check_call(['git', '-C', str(REPO), 'fetch', 'origin', 'sd21-exact-app'])
    subprocess.check_call(['git', '-C', str(REPO), 'checkout', 'sd21-exact-app'])
    subprocess.check_call(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'sd21-exact-app'])

packages = [
    'diffusers>=0.35,<1', 'transformers>=4.46,<6', 'accelerate>=1.1,<2',
    'pillow>=10,<13', 'safetensors>=0.4,<1', 'fastapi>=0.115,<1',
    'uvicorn[standard]>=0.32,<1'
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
print('✅ Projeto e dependências instalados.')

In [ ]:
# Normalmente não é necessário fazer login. Se aparecer erro 401 ou 403,
# mude o valor para True, execute esta célula e informe seu token do Hugging Face.
# O token não é colocado no projeto nem enviado ao GitHub.
FAZER_LOGIN = False
if FAZER_LOGIN:
    from huggingface_hub import notebook_login
    notebook_login()
else:
    print('Login ignorado. Altere FAZER_LOGIN somente se o Hugging Face solicitar.')

In [ ]:
import os
import queue
import re
import stat
import threading
import time
import urllib.request
from IPython.display import HTML, display

import torch

BACKEND = REPO / 'gerador-sd21' / 'backend'
CLOUDFLARED = Path('/content/cloudflared')
LOG_FILE = Path('/content/gerador-sd21.log')

if not CLOUDFLARED.exists():
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        CLOUDFLARED
    )
    CLOUDFLARED.chmod(CLOUDFLARED.stat().st_mode | stat.S_IEXEC)

print('Dispositivo:', 'GPU CUDA' if torch.cuda.is_available() else 'CPU')
if not torch.cuda.is_available():
    print('⚠️ Sem GPU: o carregamento e a geração poderão demorar bastante.')

server_log = LOG_FILE.open('w', encoding='utf-8')
server_process = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'server:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd=BACKEND, stdout=server_log, stderr=subprocess.STDOUT, text=True,
    env={**os.environ, 'PYTHONUNBUFFERED': '1'}
)

tunnel_process = subprocess.Popen(
    [str(CLOUDFLARED), 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

lines = queue.Queue()
def collect_tunnel_output():
    for output_line in tunnel_process.stdout:
        lines.put(output_line)
threading.Thread(target=collect_tunnel_output, daemon=True).start()

public_url = None
deadline = time.time() + 120
while time.time() < deadline and public_url is None:
    try:
        line = lines.get(timeout=1)
    except queue.Empty:
        if tunnel_process.poll() is not None:
            break
        continue
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)

if public_url is None:
    server_process.terminate()
    raise RuntimeError('Não foi possível criar o endereço temporário do servidor.')

print('⏳ Baixando e carregando modelo + filtro. Aguarde...')
ready = False
for _ in range(360):
    if server_process.poll() is not None:
        server_log.flush()
        print(LOG_FILE.read_text(encoding='utf-8')[-8000:])
        raise RuntimeError('O servidor parou durante o carregamento. Veja o erro acima.')
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=3) as response:
            if response.status == 200:
                ready = True
                break
    except Exception:
        pass
    time.sleep(5)

if not ready:
    raise RuntimeError('O modelo não ficou pronto dentro de 30 minutos. Consulte o log.')

print('✅ MODELO E FILTRO PRONTOS')
print('Cole este endereço no APK:', public_url)
display(HTML(f'<a href="{public_url}" target="_blank" style="font-size:20px">{public_url}</a>'))

## Manter funcionando

Deixe esta página do Colab aberta durante a geração. O endereço muda quando a sessão é reiniciada. Para encerrar, use **Ambiente de execução → Desconectar e excluir ambiente de execução**.

In [ ]:
# Execute somente se quiser conferir as últimas mensagens do servidor.
server_log.flush()
print(LOG_FILE.read_text(encoding='utf-8')[-6000:])